# Fine-tune Vietnamese Receipt OCR end-to-end
Attach a private source dataset containing `ml/receipt_ocr` and the Kaggle dataset `domixi1989/vietnamese-receipts-mc-ocr-2021`. The notebook also accepts a custom `annotations.jsonl` + line crops dataset. Enable one GPU and Internet for the first run.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, urllib.request

def run(*args):
    command = [str(item) for item in args]
    print('>', ' '.join(command))
    return subprocess.run(command, check=True)

run('nvidia-smi')
print('Python:', sys.version)

## Configuration
The cell prefers the MC-OCR train/validation label files and converts them automatically. Otherwise it discovers one `annotations.jsonl`. If candidates are ambiguous, set the paths manually.

In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working/receipt-ocr-work')
MODULE_DIR = Path('/kaggle/working/receipt-ocr-source')
PADDLEOCR_DIR = Path('/kaggle/working/PaddleOCR')
PADDLEOCR_TAG = 'v3.7.0'
PADDLE_VERSION = '3.2.0'
PADDLE_INDEX_URL = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
PRETRAINED_URL = 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams'
PRETRAINED_MODEL = WORK_DIR / 'pretrained/PP-OCRv5_mobile_rec_pretrained.pdparams'
EPOCHS = 30
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
MAX_TEXT_LENGTH = 64
EVAL_BATCH_STEP = 200
MODEL_VERSION = '0.1.0'

source_candidates = [path.parent for path in KAGGLE_INPUT.rglob('pyproject.toml') if path.parent.name == 'receipt_ocr']
if len(source_candidates) != 1:
    raise RuntimeError(f'Expected one receipt_ocr source, found: {source_candidates}')
SOURCE_MODULE_DIR = source_candidates[0]
mcocr_train_candidates = list(KAGGLE_INPUT.rglob('text_recognition_train_data.txt'))
mcocr_val_candidates = list(KAGGLE_INPUT.rglob('text_recognition_val_data.txt'))
annotation_candidates = [path for path in KAGGLE_INPUT.rglob('annotations.jsonl') if 'examples' not in path.parts]
if len(mcocr_train_candidates) == 1 and len(mcocr_val_candidates) == 1:
    MCOCR_TRAIN_LABELS = mcocr_train_candidates[0]
    MCOCR_VAL_LABELS = mcocr_val_candidates[0]
    DATASET_DIR = Path(os.path.commonpath([MCOCR_TRAIN_LABELS.resolve(), MCOCR_VAL_LABELS.resolve()]))
    ANNOTATIONS = None
elif len(annotation_candidates) == 1:
    MCOCR_TRAIN_LABELS = MCOCR_VAL_LABELS = None
    ANNOTATIONS = annotation_candidates[0]
    DATASET_DIR = ANNOTATIONS.parent
else:
    raise RuntimeError(f'Ambiguous dataset: MC-OCR train={mcocr_train_candidates}, val={mcocr_val_candidates}, annotations={annotation_candidates}')

WORK_DIR.mkdir(parents=True, exist_ok=True)
if MODULE_DIR.exists():
    shutil.rmtree(MODULE_DIR)
shutil.copytree(SOURCE_MODULE_DIR, MODULE_DIR)
print('Source:', SOURCE_MODULE_DIR, '->', MODULE_DIR)
print('Dataset:', DATASET_DIR, 'format:', 'MC-OCR' if ANNOTATIONS is None else 'JSONL')

## Install pinned runtime and obtain PaddleOCR source/weights

In [ ]:
run(sys.executable, '-m', 'pip', 'install', '-q', f'paddlepaddle-gpu=={PADDLE_VERSION}', '-i', PADDLE_INDEX_URL)
run(sys.executable, '-m', 'pip', 'install', '-q', '-e', MODULE_DIR)
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', MODULE_DIR / 'requirements-kaggle.txt')
if not PADDLEOCR_DIR.exists():
    run('git', 'clone', '--depth', '1', '--branch', PADDLEOCR_TAG, 'https://github.com/PaddlePaddle/PaddleOCR.git', PADDLEOCR_DIR)
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', PADDLEOCR_DIR / 'requirements.txt')
PRETRAINED_MODEL.parent.mkdir(parents=True, exist_ok=True)
if not PRETRAINED_MODEL.exists():
    urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_MODEL)
print('Pretrained bytes:', PRETRAINED_MODEL.stat().st_size)

## Environment and dataset preflight

In [ ]:
if ANNOTATIONS is None:
    CONVERTED_DIR = WORK_DIR / 'mcocr-converted'
    run(sys.executable, MODULE_DIR / 'scripts/convert_mcocr_kaggle.py', '--dataset-root', DATASET_DIR, '--train-labels', MCOCR_TRAIN_LABELS, '--validation-labels', MCOCR_VAL_LABELS, '--max-text-length', MAX_TEXT_LENGTH, '--output', CONVERTED_DIR)
    ANNOTATIONS = CONVERTED_DIR / 'annotations.jsonl'
    print((CONVERTED_DIR / 'mcocr_conversion_manifest.json').read_text(encoding='utf-8'))
run(sys.executable, MODULE_DIR / 'scripts/check_kaggle_environment.py', '--paddleocr-dir', PADDLEOCR_DIR, '--expected-paddleocr-tag', PADDLEOCR_TAG, '--pretrained-model', PRETRAINED_MODEL, '--dataset-dir', DATASET_DIR, '--work-dir', WORK_DIR, '--output', WORK_DIR / 'reports/environment.json')
run(sys.executable, MODULE_DIR / 'scripts/preflight_dataset.py', '--annotations', ANNOTATIONS, '--image-root', DATASET_DIR, '--max-text-length', MAX_TEXT_LENGTH, '--output', WORK_DIR / 'reports/dataset-preflight.json')
run(sys.executable, '-m', 'unittest', 'discover', '-s', MODULE_DIR / 'tests', '-v')

## Prepare reproducible splits and charset

In [ ]:
PREPARED_DIR = WORK_DIR / 'prepared'
run(sys.executable, MODULE_DIR / 'scripts/prepare_dataset.py', '--annotations', ANNOTATIONS, '--image-root', DATASET_DIR, '--output', PREPARED_DIR)
print((PREPARED_DIR / 'dataset_manifest.json').read_text(encoding='utf-8'))

## Official baseline on the locked test split

In [ ]:
PREDICTIONS_DIR = WORK_DIR / 'predictions'
REPORTS_DIR = WORK_DIR / 'reports'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
run(sys.executable, MODULE_DIR / 'scripts/predict_paddleocr.py', '--samples', PREPARED_DIR / 'samples.jsonl', '--model-name', 'PP-OCRv5_mobile_rec', '--split', 'test', '--device', 'gpu:0', '--batch-size', EVAL_BATCH_SIZE, '--output', PREDICTIONS_DIR / 'baseline-test.jsonl')
run(sys.executable, MODULE_DIR / 'scripts/evaluate_predictions.py', '--ground-truth', PREPARED_DIR / 'samples.jsonl', '--predictions', PREDICTIONS_DIR / 'baseline-test.jsonl', '--split', 'test', '--output', REPORTS_DIR / 'baseline-test-metrics.json')

## Resolve config, dry-run, then fine-tune

In [ ]:
TRAINING_DIR = WORK_DIR / 'training'
train_args = [sys.executable, MODULE_DIR / 'scripts/train_paddleocr.py', '--paddleocr-dir', PADDLEOCR_DIR, '--prepared-dir', PREPARED_DIR, '--pretrained-model', PRETRAINED_MODEL, '--output', TRAINING_DIR, '--epochs', EPOCHS, '--train-batch-size', TRAIN_BATCH_SIZE, '--eval-batch-size', EVAL_BATCH_SIZE, '--max-text-length', MAX_TEXT_LENGTH, '--eval-batch-step', EVAL_BATCH_STEP, '--num-workers', 2]
run(*train_args, '--dry-run')
print((TRAINING_DIR / 'resolved_train_config.yml').read_text(encoding='utf-8'))
run(*train_args)

## Export and smoke-test the fine-tuned model

In [ ]:
INFERENCE_DIR = WORK_DIR / 'inference-model'
run(sys.executable, MODULE_DIR / 'scripts/export_paddleocr.py', '--paddleocr-dir', PADDLEOCR_DIR, '--prepared-dir', PREPARED_DIR, '--checkpoint', TRAINING_DIR / 'best_accuracy', '--max-text-length', MAX_TEXT_LENGTH, '--output', INFERENCE_DIR)
run(sys.executable, MODULE_DIR / 'scripts/predict_paddleocr.py', '--samples', PREPARED_DIR / 'samples.jsonl', '--model-dir', INFERENCE_DIR, '--split', 'validation', '--device', 'gpu:0', '--batch-size', 8, '--limit', 8, '--output', PREDICTIONS_DIR / 'smoke-validation.jsonl')
print((PREDICTIONS_DIR / 'smoke-validation.jsonl').read_text(encoding='utf-8'))

## Full test evaluation and release gate

In [ ]:
FINE_TUNED_PREDICTIONS = PREDICTIONS_DIR / 'fine-tuned-test.jsonl'
FINE_TUNED_METRICS = REPORTS_DIR / 'fine-tuned-test-metrics.json'
run(sys.executable, MODULE_DIR / 'scripts/predict_paddleocr.py', '--samples', PREPARED_DIR / 'samples.jsonl', '--model-dir', INFERENCE_DIR, '--split', 'test', '--device', 'gpu:0', '--batch-size', EVAL_BATCH_SIZE, '--output', FINE_TUNED_PREDICTIONS)
run(sys.executable, MODULE_DIR / 'scripts/evaluate_predictions.py', '--ground-truth', PREPARED_DIR / 'samples.jsonl', '--predictions', FINE_TUNED_PREDICTIONS, '--split', 'test', '--output', FINE_TUNED_METRICS, '--max-cer', 0.15, '--min-exact-match', 0.70, '--min-coverage', 1.0)

## Package model, provenance and checksum

In [ ]:
RELEASE_DIR = WORK_DIR / 'releases'
RELEASE_ZIP = RELEASE_DIR / f'hoadon-receipt-ocr-{MODEL_VERSION}.zip'
run(sys.executable, MODULE_DIR / 'scripts/package_model_release.py', '--model-version', MODEL_VERSION, '--model-dir', INFERENCE_DIR, '--charset', PREPARED_DIR / 'vietnamese_receipt_charset.txt', '--dataset-manifest', PREPARED_DIR / 'dataset_manifest.json', '--metrics', FINE_TUNED_METRICS, '--output', RELEASE_ZIP)
for path in sorted(RELEASE_DIR.iterdir()):
    print(path.name, path.stat().st_size, 'bytes')

Download both the ZIP and `.sha256`, or publish `/kaggle/working/receipt-ocr-work/releases` as a private Kaggle Dataset before the session expires.